# Test minimal du runtime Google Colab — Oméga

Ce notebook vérifie l'environnement réellement fourni par Colab. Il ne lance aucun calcul de recherche Oméga et n'installe aucun paquet. Les résultats ne sont valides qu'après exécution effective.

## Mode d'emploi
1. Ouvrir ce notebook dans Google Colab.
2. Dans **Environnement d'exécution → Modifier le type d'exécution**, choisir GPU si cette option est disponible (sinon laisser CPU).
3. Cliquer sur **Exécuter tout**.
4. La dernière cellule écrit `omega_colab_smoke_test.json` dans `/content/`.

Aucun secret ni jeton GitHub n'est demandé ou stocké dans ce notebook. Le transfert automatique des résultats vers GitHub sera configuré et testé séparément.

In [ ]:
import sys, os, json, platform, subprocess
from datetime import datetime, timezone

print('Python :', sys.version.split()[0])
print('Plateforme :', platform.platform())
print('Répertoire courant :', os.getcwd())


In [ ]:
gpu = {'detected': False, 'name': None, 'probe': 'not_run'}
try:
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    if probe.returncode == 0 and probe.stdout.strip():
        gpu.update({'detected': True, 'name': probe.stdout.strip().splitlines()[0], 'probe': 'nvidia-smi'})
    else:
        gpu['probe'] = 'nvidia-smi_unavailable_or_no_gpu'
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    gpu['probe'] = type(exc).__name__

try:
    import torch
    gpu['pytorch_available'] = True
    gpu['cuda_available'] = bool(torch.cuda.is_available())
    if gpu['cuda_available']:
        gpu['torch_device_name'] = torch.cuda.get_device_name(0)
except ImportError:
    gpu['pytorch_available'] = False
    gpu['cuda_available'] = False

print(json.dumps(gpu, indent=2, ensure_ascii=False))


In [ ]:
# Test arithmétique déterministe, sans dépendance externe
n = 100_000
total = sum(range(n))
expected = n * (n - 1) // 2
assert total == expected, f'Erreur : {total} != {expected}'
assert total == 4_999_950_000
print('TEST EXACT : RÉUSSI')
print('Somme calculée :', total)
print('Valeur attendue :', expected)


In [ ]:
result = {
    'test': 'omega_colab_runtime_smoke_test',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version,
    'platform': platform.platform(),
    'gpu': gpu,
    'arithmetic_test': {
        'status': 'PASS',
        'n': n,
        'computed': total,
        'expected': expected
    }
}
output_path = '/content/omega_colab_smoke_test.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print('Résultat enregistré :', output_path)
print(json.dumps(result, indent=2, ensure_ascii=False))
